# Validation Annotation Ingestion

Ingests the manual Label Studio annotation of `eval_worklist.csv` (written by
notebook 04_a) and writes the structured ground-truth table notebook evaluation
scores against.

**Reads:** `data/annotation_sample/eval_worklist.csv` (04_a) and the Label
Studio export `data/processed/eval_labelstudio.json` (manual — exported by
hand from the Label Studio UI, no notebook produces it).

**Writes:** `data/annotations/labels_val.csv`, brush masks decoded to
`data/annotations/masks/`.

## Annotation steps (do this in Label Studio before running the cells below)

1. one **presentation-type label** from the 7 classes;
2. a **brush mask** over the colour region for every image except `unclassifiable`
   (yes, including `swatch` — it has its own segmenter).

Ground-truth CIELAB comes from the median color inside this brush mask, applied
to the original image — no separate manually-cropped swatch (unlike notebook
03_b's training-set ground truth). Export the Label Studio project JSON to
`data/processed/eval_labelstudio.json` (or a `project-*-eval-*.json` file). The
cells below stay inert until that export exists.

In [1]:
import os
import re
import json
import glob
import hashlib
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from skimage.color import rgb2lab

from label_studio_converter.brush import decode_rle as _ls_decode_rle

warnings.filterwarnings('ignore')

# Notebook runs from notebooks/ ; paths mirror notebook 04_a
BASE     = os.path.abspath('../data')
DATA     = os.path.join(BASE, 'processed')
ANN_DIR  = os.path.join(BASE, 'annotations')
ANN_SAM  = os.path.join(BASE, 'annotation_sample')
MASKS_DIR = os.path.join(ANN_DIR, 'masks')
CLEAN    = os.path.join(BASE, 'img', 'original_clean')

# Local Label Studio install's media store (macOS default). Used only to
# disambiguate the filename-truncation bug — see CLAUDE.md — by hashing pixel
# content; harmless if it doesn't exist on this machine (falls back gracefully).
LS_MEDIA_ROOT = os.path.expanduser('~/Library/Application Support/label-studio/media/upload')

os.makedirs(MASKS_DIR, exist_ok=True)

/Users/connie/dev/lipstick_color_extraction/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def find_image(img_name, img_dirs=(CLEAN,)):
    for d in img_dirs:
        p = os.path.join(d, img_name)
        if os.path.exists(p):
            return p
    return None


def resolve_img_name(img_name, img_dirs=(CLEAN,)):
    """Map a Label-Studio img_name to the actual file on disk (strip a 7-char
    dedup suffix if needed). From notebook 06."""
    if find_image(img_name, img_dirs) is not None:
        return img_name
    base = re.sub(r'_[A-Za-z0-9]{7}(\.jpg)$', r'\1', img_name)
    if base != img_name and find_image(base, img_dirs) is not None:
        return base
    return None


def decode_rle(rle, height, width):
    """Label-Studio brush RLE -> binary mask (H x W uint8). From notebook 06."""
    flat = _ls_decode_rle(rle)
    rgba = np.array(flat).reshape(height, width, 4)
    return (rgba[:, :, 3] > 0).astype(np.uint8)


def extract_cielab_from_mask(img_name_disk, mask_path):
    """Ground-truth CIELAB: median color inside the annotator's own brush mask,
    applied to the original image. Same masked-color-extraction method used for
    the *predicted* color in notebook 07, so true vs. predicted ΔE compares
    like with like. Returns np.array([L, a, b]) or None if there's no mask or
    it's empty."""
    if not isinstance(mask_path, str) or not os.path.exists(mask_path):
        return None
    img_path = find_image(img_name_disk)
    if img_path is None:
        return None
    img = np.array(Image.open(img_path).convert('RGB'))
    mask = (np.array(Image.open(mask_path).convert('L')) > 127).astype(np.uint8)
    if mask.shape != img.shape[:2]:
        mask = np.array(Image.fromarray(mask * 255).resize(img.shape[1::-1], Image.NEAREST)) // 255
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        lab = rgb2lab(img / 255.0)
    px = lab[mask == 1]
    return None if len(px) == 0 else np.median(px, axis=0)


def _content_hash(path):
    with Image.open(path) as im:
        return hashlib.md5(im.convert('RGB').tobytes()).hexdigest()


def find_truncation_candidates(trunc_name, worklist_names):
    """Worklist img_name_disk values that could have produced this Label-Studio
    truncated stem: same prefix, before the shade name got cut off (see CLAUDE.md
    — Label Studio truncates long filenames on upload)."""
    stem = re.sub(r'_[A-Za-z0-9]{7}(\.jpg)$', r'\1', trunc_name)[:-4]
    return [n for n in worklist_names if n.startswith(stem)]


def resolve_by_content(upload_path, worklist, candidates, img_dir=CLEAN):
    """Disambiguate a truncated Label Studio filename by matching pixel content
    (MD5 of decoded RGB bytes) against the candidate worklist images that share
    its truncated stem. Returns the matching img_name_disk, or None if the LS
    media store isn't reachable or content matches zero / more than one candidate."""
    if not os.path.exists(upload_path):
        return None
    target = _content_hash(upload_path)
    matches = []
    for name in candidates:
        p = os.path.join(img_dir, name)
        if os.path.exists(p) and _content_hash(p) == target:
            matches.append(name)
    return matches[0] if len(matches) == 1 else None


# Label Studio's default export name doesn't encode "eval" — it's
# `project-<id>-at-<timestamp>-<hash>.json`. This project also accumulates
# training-set annotations over time, so pick the most recently modified
# export and rely on EVAL_ANNOTATION_CUTOFF (next cell) to isolate the
# validation-round annotations from older training ones.
_export_candidates = sorted(
    glob.glob(os.path.join(DATA, 'eval_labelstudio.json'))
    + glob.glob(os.path.join(DATA, 'project-*.json')),
    key=os.path.getmtime, reverse=True,
)
EVAL_EXPORT = _export_candidates[0] if _export_candidates else os.path.join(DATA, 'eval_labelstudio.json')

worklist = pd.read_csv(os.path.join(ANN_SAM, 'eval_worklist.csv'))
print(f'worklist rows: {len(worklist)}')
print(f'looking for Label Studio export at: {EVAL_EXPORT}  '
      f'({"found" if os.path.exists(EVAL_EXPORT) else "not found yet"})')

worklist rows: 503
looking for Label Studio export at: /Users/connie/dev/lipstick_color_extraction/data/processed/project-4-at-2026-09-18-21-20-25213916.json  (found)


In [3]:
LABEL_REMAP = {'multi_impage': 'unclassifiable', 'color_not_shown': 'unclassifiable'}

# This Label Studio project also holds older training-set annotations
# (notebook 03_c). Keep only the validation round, annotated from this date.
EVAL_ANNOTATION_CUTOFF = '2026-09-01'

if not os.path.exists(EVAL_EXPORT):
    print(f'{EVAL_EXPORT} not found — skipped. Annotate the worklist, export, re-run.')
    ls_labels = None
else:
    with open(EVAL_EXPORT) as f:
        raw = json.load(f)
    recs = []
    n_skipped_old = 0
    n_resolved_by_content = 0
    n_unresolved = 0
    worklist_names = worklist['img_name_disk']
    for task in raw:
        raw_name = os.path.basename(task['data']['image'])
        img_name = raw_name.split('-', 1)[-1] if '-' in raw_name else raw_name
        anns = task.get('annotations') or []
        if not anns or not anns[0].get('result'):
            continue
        if anns[0].get('created_at', '') < EVAL_ANNOTATION_CUTOFF:
            n_skipped_old += 1
            continue
        res = anns[0]['result']
        brush = next((r for r in res if r['value'].get('rle')), None)
        label = None
        for r in res:
            v = r['value']
            if v.get('brushlabels'):
                label = v['brushlabels'][0]
            elif v.get('choices'):
                label = v['choices'][0]
        if label is None:
            continue
        label = LABEL_REMAP.get(label, label)
        disk = resolve_img_name(os.path.splitext(img_name)[0] + '.jpg')
        if disk is None:
            # Label Studio truncated this filename on upload (see CLAUDE.md) —
            # disambiguate by pixel content against worklist rows sharing the
            # same truncated stem, using LS's own local media store.
            candidates = find_truncation_candidates(img_name, worklist_names)
            if candidates:
                path_parts = task['data']['image'].strip('/').split('/')
                media_subdir = path_parts[-2] if len(path_parts) >= 2 else ''
                upload_path = os.path.join(LS_MEDIA_ROOT, media_subdir, raw_name)
                disk = resolve_by_content(upload_path, worklist, candidates)
            if disk is not None:
                n_resolved_by_content += 1
        if disk is None:
            n_unresolved += 1
            disk = os.path.splitext(img_name)[0] + '.jpg'
        mask_path = os.path.join(MASKS_DIR, os.path.splitext(disk)[0] + '.png')
        if brush is not None:
            h = brush.get('original_height') or brush.get('value', {}).get('original_height')
            w = brush.get('original_width') or brush.get('value', {}).get('original_width')
            if h and w:
                m = decode_rle(brush['value']['rle'], int(h), int(w))
                Image.fromarray(m * 255).save(mask_path)
        recs.append({'img_name': img_name, 'img_name_disk': disk, 'label': label,
                     'mask_path': mask_path if brush is not None else np.nan})
    ls_labels = pd.DataFrame(recs)
    ls_labels = ls_labels.merge(worklist[['img_name_disk', 'block', 'duplicate_group_id']], on='img_name_disk', how='left')
    _unknown = ls_labels['block'].isna().sum()
    if _unknown:
        print(f'warning: {_unknown} annotated images are not in the worklist — dropped')
        ls_labels = ls_labels[ls_labels['block'].notna()].copy()
    print(f'{n_skipped_old} pre-{EVAL_ANNOTATION_CUTOFF} annotations skipped (training-set annotations in the same project)')
    print(f'{n_resolved_by_content} filenames disambiguated by pixel-content hash '
          f'(Label Studio truncation — see CLAUDE.md)')
    if n_unresolved:
        print(f'warning: {n_unresolved} truncated filenames could not be resolved '
              f'to a worklist image even by content hash')
    print(f'{len(ls_labels)} eval annotations parsed')
    print(ls_labels.groupby('label').size().to_string())

379 pre-2026-09-01 annotations skipped (training-set annotations in the same project)
58 filenames disambiguated by pixel-content hash (Label Studio truncation — see CLAUDE.md)
487 eval annotations parsed
label
bullet            128
closed             15
lips               14
liquid            103
pencil             24
swatch            171
unclassifiable     32


In [4]:
if ls_labels is None:
    print('skipped — no annotations yet.')
else:
    def _lab(row):
        # unclassifiable has no coherent color region by convention (CLAUDE.md)
        # even if a mask happens to exist for it — never extract a color from one.
        if row['label'] == 'unclassifiable':
            v = None
        else:
            v = extract_cielab_from_mask(row['img_name_disk'], row['mask_path'])
        return pd.Series([np.nan] * 3 if v is None else list(v), index=['true_L', 'true_a', 'true_b'])

    ls_labels = pd.concat([ls_labels, ls_labels.apply(_lab, axis=1)], axis=1)
    _cov = ls_labels['true_L'].notna().sum()
    print(f'ground-truth CIELAB extracted from mask: {_cov} / {len(ls_labels)}')
    print('  (unclassifiable and images with no mask leave ΔE undefined)')

ground-truth CIELAB extracted from mask: 455 / 487
  (unclassifiable and images with no mask leave ΔE undefined)


### Drop redundant duplicate images

Some worklist images are pixel-identical (`duplicate_group_id` from notebook
04_a) because the seller uploaded the same photo for multiple shades. Keeping
every annotated copy in the evaluation set would double-count the same photo
as if it were independent evidence, without adding any real coverage. Keep one
representative annotation per `duplicate_group_id` (the one with a mask, if
any group member has one) and drop the rest.


In [5]:
if ls_labels is None:
    print('skipped — no annotations yet.')
else:
    dup_mask = ls_labels['duplicate_group_id'].notna()

    def _pick_representative(g):
        with_mask = g[g['mask_path'].notna()]
        return with_mask.iloc[[0]] if len(with_mask) else g.iloc[[0]]

    deduped = ls_labels[dup_mask].groupby('duplicate_group_id', group_keys=False).apply(_pick_representative)
    n_dropped = int(dup_mask.sum() - len(deduped))
    ls_labels = pd.concat([ls_labels[~dup_mask], deduped], ignore_index=True)
    print(f'{n_dropped} redundant duplicate-image rows dropped '
          f'(kept 1 representative per duplicate_group_id)')


6 redundant duplicate-image rows dropped (kept 1 representative per duplicate_group_id)


In [6]:
LABELS_SCHEMA = ['img_name', 'label', 'img_name_disk', 'true_L', 'true_a', 'true_b', 'mask_path', 'block']

if ls_labels is None:
    labels_val_df = None
    print('skipped — labels_val.csv not written.')
else:
    labels_val_df = ls_labels.reindex(columns=LABELS_SCHEMA).reset_index(drop=True)
    labels_val_df.to_csv(os.path.join(ANN_DIR, 'labels_val.csv'), index=False)
    print(f'wrote labels_val.csv ({len(labels_val_df)} rows)')

wrote labels_val.csv (481 rows)


## How to run

```bash
.venv/bin/python -m jupyter nbconvert --to notebook --execute --inplace \
  notebooks/04_b_validation_annotation_image.ipynb
```

Run once `eval_worklist.csv` (from 04_a) has been annotated in Label Studio and
exported. On a run without an export yet, every cell just prints "skipped" and
does nothing — that's expected.